In [ ]:
import os
import json
import re
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter
from wordcloud import WordCloud
import pathlib

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    classification_report, f1_score, accuracy_score, roc_auc_score,
    confusion_matrix, ConfusionMatrixDisplay
)
from sklearn.preprocessing import label_binarize

import tensorflow as tf
from tensorflow.keras import layers
import xgboost as xgb
import transformers
import torch
from transformers import BertTokenizer
from sentence_transformers import SentenceTransformer

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
tf.random.set_seed(SEED)

DATA_DIR = pathlib.Path("../data")
REPORTS_DIR = pathlib.Path("../reports")
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

CLASS_NAMES = ["non-toxic", "toxic"]
N_CLASSES = 2

print(f"Python: {os.sys.version}")
print(f"NumPy: {np.__version__}")
print(f"Pandas: {pd.__version__}")
print(f"Scikit-learn: {__import__('sklearn').__version__}")
print(f"TensorFlow: {tf.__version__}")
print(f"XGBoost: {xgb.__version__}")
print(f"Transformers: {transformers.__version__}")
print(f"PyTorch: {torch.__version__}")
print(f"Matplotlib: {__import__('matplotlib').__version__}")
print(f"Seaborn: {sns.__version__}")
print("=" * 50)
print(f"SEED: {SEED}")
print("Konfiguracja zakończona")

## 1. EDA - Eksploracyjna Analiza Danych

In [ ]:
df_raw = pd.read_csv(DATA_DIR / "raw/train.csv")
print(f"Rozmiar surowych danych: {df_raw.shape}")
df_raw.head()

In [ ]:
labels = ["toxic", "severe_toxic", "obscene", "threat", "insult", "identity_hate"]

print("Rozkład etykiet w surowych danych:")
for label in labels:
    count = df_raw[label].sum()
    pct = df_raw[label].mean() * 100
    print(f"  {label}: {count} ({pct:.1f}%)")

df_raw["is_toxic"] = df_raw[labels].max(axis=1)
toxic_count = df_raw["is_toxic"].sum()
print(f"\nŁącznie komentarzy toksycznych: {toxic_count} ({toxic_count/len(df_raw)*100:.1f}%)")

In [ ]:
print("Przykładowe komentarze toksyczne:")
toxic_samples = df_raw[df_raw["is_toxic"] == 1]["comment_text"].sample(3, random_state=SEED)
for i, text in enumerate(toxic_samples, 1):
    print(f"{i}. {text[:100]}...")

print("\nPrzykładowe komentarze nietoksyczne:")
nontoxic_samples = df_raw[df_raw["is_toxic"] == 0]["comment_text"].sample(3, random_state=SEED)
for i, text in enumerate(nontoxic_samples, 1):
    print(f"{i}. {text[:100]}...")

In [ ]:
df_raw["text_length"] = df_raw["comment_text"].str.len()

print("Statystyki długości komentarzy:")
print(df_raw["text_length"].describe())

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.hist(df_raw["text_length"], bins=50, alpha=0.7)
ax1.set_title("Rozkład długości komentarzy")
ax1.set_xlabel("Długość tekstu")
ax1.set_ylabel("Liczba komentarzy")

ax2.boxplot([df_raw[df_raw["is_toxic"]==0]["text_length"],
             df_raw[df_raw["is_toxic"]==1]["text_length"]],
            labels=["Non-toxic", "Toxic"])
ax2.set_title("Porównanie długości")
ax2.set_ylabel("Długość tekstu")

plt.tight_layout()
plt.show()

In [ ]:
def clean_text(text):
    text = re.sub(r"[^a-zA-Z\s]", "", text.lower())
    return text

toxic_texts = df_raw[df_raw["is_toxic"] == 1]["comment_text"].apply(clean_text)
nontoxic_texts = df_raw[df_raw["is_toxic"] == 0]["comment_text"].apply(clean_text)

toxic_words = " ".join(toxic_texts).split()
nontoxic_words = " ".join(nontoxic_texts).split()

toxic_freq = Counter(toxic_words).most_common(20)
nontoxic_freq = Counter(nontoxic_words).most_common(20)

fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(2, 2, figsize=(15, 10))

words, counts = zip(*toxic_freq)
ax1.barh(words, counts)
ax1.set_title("Najczęstsze słowa - komentarze toksyczne")

words, counts = zip(*nontoxic_freq)
ax2.barh(words, counts)
ax2.set_title("Najczęstsze słowa - komentarze nietoksyczne")

toxic_text = " ".join(toxic_words[:10000])
wordcloud_toxic = WordCloud(width=400, height=300, background_color="white",
                           max_words=100).generate(toxic_text)
ax3.imshow(wordcloud_toxic, interpolation="bilinear")
ax3.axis("off")
ax3.set_title("Chmura słów - toksyczne")

nontoxic_text = " ".join(nontoxic_words[:10000])
wordcloud_nontoxic = WordCloud(width=400, height=300, background_color="white",
                              max_words=100).generate(nontoxic_text)
ax4.imshow(wordcloud_nontoxic, interpolation="bilinear")
ax4.axis("off")
ax4.set_title("Chmura słów - nietoksyczne")

plt.tight_layout()
plt.show()

## 2. Przygotowanie Danych i Split

In [ ]:
df_all = pd.read_csv(DATA_DIR / "raw/train.csv").dropna().drop_duplicates()

labels = ["toxic", "severe_toxic", "obscene", "threat", "insult", "identity_hate"]
df_all["toxic_level"] = (df_all[labels].max(axis=1) > 0).astype(int)

df_all = df_all.drop(["id"] + labels, axis=1)

print(f"Po przetworzeniu: {df_all.shape}")
print("Rozkład klas:")
print(df_all["toxic_level"].value_counts().sort_index())

In [ ]:
import nltk
from nltk.corpus import stopwords

try:
    sw = stopwords.words("english")
except:
    nltk.download("stopwords")
    sw = stopwords.words("english")

def preprocess_text(text):
    text = re.sub(r'https?://\S+', "", text)
    text = text.lower()
    text = re.sub(r"'", "", text)
    text = re.sub(r"[^a-z]+", " ", text)
    words = [w for w in text.split() if w not in sw][:1024]
    return " ".join(words).strip()

df_all["comment_text"] = df_all["comment_text"].apply(preprocess_text)
print("Tekst wyczyszczony")

In [ ]:
df_train_full, df_test = train_test_split(df_all, test_size=5000, stratify=df_all["toxic_level"], random_state=SEED)
df_train_rest, df_val = train_test_split(df_train_full, test_size=2000, stratify=df_train_full["toxic_level"], random_state=SEED)

df_toxic = df_train_rest[df_train_rest["toxic_level"] == 1]
df_nontoxic = df_train_rest[df_train_rest["toxic_level"] == 0]

n_samples = 10000
df_nontoxic_sample = df_nontoxic.sample(n=n_samples, random_state=SEED)

replace = len(df_toxic) < n_samples
df_toxic_sample = df_toxic.sample(n=n_samples, replace=replace, random_state=SEED)

df_train = pd.concat([df_nontoxic_sample, df_toxic_sample]).sample(frac=1, random_state=SEED).reset_index(drop=True)

df_train.to_csv(DATA_DIR / "df_train.csv", index=False)
df_val.to_csv(DATA_DIR / "df_val.csv", index=False)
df_test.to_csv(DATA_DIR / "df_test.csv", index=False)

print(f"Train (balanced): {len(df_train)}, Val: {len(df_val)}, Test: {len(df_test)}")
print("\nRozkład klas w train:")
print(df_train["toxic_level"].value_counts().sort_index())
print("\nRozkład klas w val:")
print(df_val["toxic_level"].value_counts().sort_index())
print("\nRozkład klas w test:")
print(df_test["toxic_level"].value_counts().sort_index())

## 3. Przygotowanie Sample Testowego (500 próbek)

In [ ]:
df_test_sample = df_test.copy()
df_test_sample = df_test_sample.reset_index(drop=True)
df_test_sample['original_idx'] = df_test_sample.index

df_test_sample.to_csv(DATA_DIR / "df_test_sample.csv", index=False)

sample_indices = df_test_sample.index.tolist()
with open(DATA_DIR / "sample_indices.txt", "w") as f:
    f.write(str(sample_indices))

print(f"Zbiór testowy: {len(df_test_sample)} próbek")

## 4. Modele

In [ ]:
df_train = pd.read_csv(DATA_DIR / "df_train.csv")
df_val = pd.read_csv(DATA_DIR / "df_val.csv")
df_test_sample = pd.read_csv(DATA_DIR / "df_test_sample.csv")

X_train = df_train['comment_text'].fillna('')
y_train = df_train['toxic_level'].values

X_val = df_val['comment_text'].fillna('')
y_val = df_val['toxic_level'].values

X_test = df_test_sample['comment_text'].fillna('')
y_test = df_test_sample['toxic_level'].values

print(f"Dane przygotowane: train={len(X_train)}, val={len(X_val)}, test={len(X_test)}")

### 4.1 Baseline: TF-IDF + Logistic Regression

In [ ]:
tfidf = TfidfVectorizer(
    lowercase=True,
    strip_accents="unicode",
    ngram_range=(1, 2),
    min_df=2,
    max_features=200_000,
)

clf_baseline = LogisticRegression(
    solver="lbfgs",
    C=2.0,
    max_iter=1000,
    random_state=SEED,
)

pipe_baseline = Pipeline([("tfidf", tfidf), ("clf", clf_baseline)])
pipe_baseline.fit(X_train, y_train)

In [ ]:
y_pred_baseline = pipe_baseline.predict(X_test)
y_prob_baseline = pipe_baseline.predict_proba(X_test)

print(f"Unikalne predykcje: {np.unique(y_pred_baseline, return_counts=True)}")

### 4.2 XGBoost z Embeddings

In [ ]:
print("Ładowanie modelu sentence-transformers...")
st_model = SentenceTransformer('all-MiniLM-L6-v2')

def get_embeddings(texts):
    return st_model.encode(texts, show_progress_bar=True)

print("Generowanie embeddingów dla zbioru treningowego...")
X_train_emb = get_embeddings(X_train.tolist())

print("Generowanie embeddingów dla zbioru testowego...")
X_test_emb = get_embeddings(X_test.tolist())

print(f"Kształt X_train_emb: {X_train_emb.shape}")
print(f"Kształt X_test_emb: {X_test_emb.shape}")

In [ ]:
dtrain = xgb.DMatrix(X_train_emb, label=y_train)
dtest = xgb.DMatrix(X_test_emb, label=y_test)

param = {
    'max_depth': 6,
    'eta': 0.1,
    'objective': 'binary:logistic',
    'eval_metric': 'logloss',
    'nthread': 4,
    'seed': SEED,
    'subsample': 0.8,
    'colsample_bytree': 0.8
}

num_round = 100
bst = xgb.train(param, dtrain, num_round, verbose_eval=False)

In [ ]:
y_prob_xgb = bst.predict(dtest)
y_pred_xgb = (y_prob_xgb > 0.5).astype(int)

print(f"Unikalne predykcje: {np.unique(y_pred_xgb, return_counts=True)}")

### 4.3 BERT

### 4.3.1 Trenowanie BERT i generowanie predykcji

In [ ]:
import torch
import torch.nn as nn
from torch.optim import AdamW
from torch.utils.data import DataLoader, RandomSampler, SequentialSampler, TensorDataset
from transformers import BertTokenizer, BertForSequenceClassification, get_linear_schedule_with_warmup
import time, datetime, random, gc
from tqdm import tqdm

def BERTAify(clean_df, tokenizer, max_length):
    input_ids = []
    attention_masks = []
    for comment in clean_df["comment_text"]:
        encoded = tokenizer.encode_plus(
            str(comment),
            add_special_tokens=True,
            max_length=max_length,
            padding='max_length',
            truncation=True,
            return_attention_mask=True,
            return_tensors='pt'
        )
        input_ids.append(encoded['input_ids'])
        attention_masks.append(encoded['attention_mask'])

    input_ids = torch.cat(input_ids, dim=0)
    attention_masks = torch.cat(attention_masks, dim=0)
    labels = torch.tensor(clean_df["toxic_level"].values, dtype=torch.long)
    return TensorDataset(input_ids, attention_masks, labels)

def flat_accuracy(preds, labels):
    pred_flat = np.argmax(preds, axis=1).flatten()
    labels_flat = labels.flatten()
    return np.sum(pred_flat == labels_flat) / len(labels_flat)

def format_time(elapsed):
    elapsed_rounded = int(round(elapsed))
    return str(datetime.timedelta(seconds=elapsed_rounded))

In [ ]:
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased', do_lower_case=True)

batch_size = 32
max_length = 128
epochs = 3
lr = 2e-5

clean_train = pd.read_csv(DATA_DIR / "df_train.csv")
clean_val = pd.read_csv(DATA_DIR / "df_val.csv")
clean_test = pd.read_csv(DATA_DIR / "df_test.csv")

for df in (clean_train, clean_val, clean_test):
    df["comment_text"] = df["comment_text"].astype(str)
    df["toxic_level"] = df["toxic_level"].astype(int)

train_dataset = BERTAify(clean_train, tokenizer, max_length=max_length)
val_dataset = BERTAify(clean_val, tokenizer, max_length=max_length)
test_dataset = BERTAify(clean_test, tokenizer, max_length=max_length)

train_loader = DataLoader(train_dataset, sampler=RandomSampler(train_dataset), batch_size=batch_size)
val_loader = DataLoader(val_dataset, sampler=SequentialSampler(val_dataset), batch_size=batch_size)
test_dataloader = DataLoader(test_dataset, sampler=SequentialSampler(test_dataset), batch_size=batch_size)

print(f"Train samples: {len(train_dataset)}, Val samples: {len(val_dataset)}, Test samples: {len(test_dataset)}")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = BertForSequenceClassification.from_pretrained("bert-base-uncased", num_labels=2)
model.to(device)

optimizer = AdamW(model.parameters(), lr=lr, eps=1e-8)
total_steps = len(train_loader) * epochs
scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=0, num_training_steps=total_steps)

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

In [ ]:
training_stats = []
total_t0 = time.time()

for epoch_i in range(epochs):
    print(f"\nEpoch {epoch_i+1} / {epochs}")

    model.train()
    total_train_loss = 0
    t0 = time.time()

    train_bar = tqdm(train_loader, desc=f"Training {epoch_i+1}/{epochs}")

    for batch in train_bar:
        b_input_ids = batch[0].to(device)
        b_input_mask = batch[1].to(device)
        b_labels = batch[2].to(device)

        optimizer.zero_grad()

        outputs = model(b_input_ids, token_type_ids=None, attention_mask=b_input_mask, labels=b_labels)
        loss = outputs.loss
        total_train_loss += loss.item()

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()

        train_bar.set_postfix({"loss": f"{loss.item():.4f}"})

    avg_train_loss = total_train_loss / len(train_loader)
    training_time = format_time(time.time() - t0)
    print(f"  Average training loss: {avg_train_loss:.4f}, Time: {training_time}")

    model.eval()
    total_eval_accuracy = 0
    total_eval_loss = 0

    val_bar = tqdm(val_loader, desc=f"Validation {epoch_i+1}/{epochs}")

    for batch in val_bar:
        b_input_ids = batch[0].to(device)
        b_input_mask = batch[1].to(device)
        b_labels = batch[2].to(device)

        with torch.no_grad():
            outputs = model(b_input_ids, token_type_ids=None, attention_mask=b_input_mask, labels=b_labels)

        loss = outputs.loss
        logits = outputs.logits.detach().cpu().numpy()
        label_ids = b_labels.cpu().numpy()

        total_eval_loss += loss.item()
        total_eval_accuracy += flat_accuracy(logits, label_ids)

    avg_val_accuracy = total_eval_accuracy / len(val_loader)
    avg_val_loss = total_eval_loss / len(val_loader)
    print(f"  Validation accuracy: {avg_val_accuracy:.4f}, Loss: {avg_val_loss:.4f}")

    training_stats.append({
        "epoch": epoch_i + 1,
        "Training Loss": avg_train_loss,
        "Validation Loss": avg_val_loss,
        "Validation Accuracy": avg_val_accuracy
    })

print(f"\nTrening zakończony w {format_time(time.time() - total_t0)}")

In [ ]:
model.eval()
predictions = []

test_bar = tqdm(test_dataloader, desc="Generowanie predykcji...")
for batch in test_bar:
    b_input_ids = batch[0].to(device)
    b_input_mask = batch[1].to(device)

    with torch.no_grad():
        outputs = model(b_input_ids, token_type_ids=None, attention_mask=b_input_mask)
        logits = outputs.logits.detach().cpu().numpy()
        pred_flat = np.argmax(logits, axis=1).flatten()
        predictions.extend(pred_flat)

with open(f"BERTpred{max_length}.txt", "w") as f:
    f.write(str(predictions))

print(f"Predykcje BERT zapisane do BERTpred{max_length}.txt")
print(f"Liczba predykcji: {len(predictions)}")
print(f"Rozkład: {np.unique(predictions, return_counts=True)}")

y_pred_bert_generated = np.array(predictions)
y_pred_bert = y_pred_bert_generated[sample_indices]

In [ ]:
with open("BERTpred128.txt", "r") as f:
    bert_preds_full = eval(f.read())

with open(DATA_DIR / "sample_indices.txt", "r") as f:
    sample_indices = eval(f.read())

y_pred_bert = np.array([bert_preds_full[i] for i in sample_indices])

print(f"Predykcje BERT wyciągnięte: {len(y_pred_bert)}")
print(f"Unikalne predykcje: {np.unique(y_pred_bert, return_counts=True)}")

### 4.4 LLM: Groq API (Llama 3.1 70B)

In [ ]:
LLM_MODEL = "llama-3.1-70b-versatile"
LLM_PREDICTIONS_FILE = DATA_DIR / "llm_predictions.json"

if LLM_PREDICTIONS_FILE.exists():
    with open(LLM_PREDICTIONS_FILE, "r") as f:
        llm_cache = json.load(f)
    print(f"Załadowano {len(llm_cache)} zapisanych predykcji LLM")
else:
    llm_cache = {}
    print("Brak zapisanych predykcji")

In [ ]:
# Funkcja do klasyfikacji przez LLM
def classify_with_llm(client, text, max_retries=3):
    prompt = f"""Klasyfikuj następujący komentarz jako TOXIC lub NONTOXIC.

Zasady:
- NONTOXIC = Normalna, szanująca dyskusja
- TOXIC = Mowa nienawiści, obraźliwe, wulgarne, groźby

Odpowiedz DOKŁADNIE w formacie: LABEL=NONTOXIC lub LABEL=TOXIC

Komentarz: {text[:1500]}

Odpowiedź:"""

    label_map = {"NONTOXIC": 0, "TOXIC": 1}
    
    for attempt in range(max_retries):
        try:
            response = client.chat.completions.create(
                model=LLM_MODEL,
                messages=[{"role": "user", "content": prompt}],
                temperature=0,
                max_tokens=20,
            )
            answer = response.choices[0].message.content.strip().upper()
            
            match = re.search(r'LABEL\s*=\s*(NONTOXIC|TOXIC)', answer)
            if match:
                label = match.group(1)
                return label_map.get(label, 0)
            else:
                for label, value in label_map.items():
                    if label in answer:
                        return value
                print(f"Nieprawidłowa odpowiedź: {answer}")
                return 0
                
        except Exception as e:
            print(f"Próba {attempt+1} nieudana: {e}")
            if attempt < max_retries - 1:
                time.sleep(2 ** attempt)
    
    return 0

In [ ]:
LLM_SAMPLE_SIZE = 100
llm_indices = df_test_sample.index[:LLM_SAMPLE_SIZE]

try:
    from groq import Groq
    GROQ_AVAILABLE = True
except ImportError:
    GROQ_AVAILABLE = False
    print("Groq nie zainstalowany. Uruchom: pip install groq")

if GROQ_AVAILABLE and os.environ.get("GROQ_API_KEY"):
    client = Groq(api_key=os.environ.get("GROQ_API_KEY"))
    
    y_pred_llm = []
    
    print(f"Rozpoczynam klasyfikację LLM dla {LLM_SAMPLE_SIZE} próbek...")
    
    for i, idx in enumerate(llm_indices):
        text = X_test.iloc[i]
        cache_key = str(idx)
        
        if cache_key in llm_cache:
            pred = llm_cache[cache_key]
        else:
            pred = classify_with_llm(client, text)
            llm_cache[cache_key] = pred
            
            if (i + 1) % 25 == 0:
                time.sleep(5)
                
            if (i + 1) % 50 == 0:
                with open(LLM_PREDICTIONS_FILE, "w") as f:
                    json.dump(llm_cache, f)
                print(f"Przetworzono {i+1}/{LLM_SAMPLE_SIZE} próbek")
        
        y_pred_llm.append(pred)
    
    with open(LLM_PREDICTIONS_FILE, "w") as f:
        json.dump(llm_cache, f)
    
    y_pred_llm = np.array(y_pred_llm)
    print(f"\nPredykcje LLM wykonane: {len(y_pred_llm)}")
    print(f"Unikalne predykcje: {np.unique(y_pred_llm, return_counts=True)}")
    
else:
    print("API Groq niedostępne. Ustaw zmienną środowiskową GROQ_API_KEY.")
    y_pred_llm = None

## 5. Ewaluacja i Porównanie

In [ ]:
def calculate_metrics(y_true, y_pred, y_prob=None, name="Model"):
    """Oblicza wszystkie wymagane metryki."""
    acc = accuracy_score(y_true, y_pred)
    f1 = f1_score(y_true, y_pred, average="macro")
    
    auc = None
    if y_prob is not None:
        try:
            if len(np.unique(y_true)) <= 2:
                if y_prob.ndim > 1:
                    y_prob_pos = y_prob[:, 1]
                else:
                    y_prob_pos = y_prob
                auc = roc_auc_score(y_true, y_prob_pos)
            else:
                auc = roc_auc_score(y_true, y_prob, average="macro", multi_class="ovr")
        except Exception as e:
            print(f"Błąd liczenia AUC dla {name}: {e}")
            auc = None
    
    return {
        "model": name,
        "accuracy": acc,
        "macro_f1": f1,
        "roc_auc": auc,
        "predictions": y_pred
    }

In [ ]:
results_full = []
results_llm_subset = []

results_full.append(calculate_metrics(y_test, y_pred_baseline, y_prob_baseline, "TF-IDF + LogReg"))
results_full.append(calculate_metrics(y_test, y_pred_xgb, y_prob_xgb, "XGBoost"))
results_full.append(calculate_metrics(y_test, y_pred_bert, None, "BERT (128)"))

if y_pred_llm is not None:
    subset_slice = slice(0, len(y_pred_llm))
    y_test_subset = y_test[subset_slice]
    
    results_llm_subset.append(calculate_metrics(y_test_subset, y_pred_baseline[subset_slice], None, "TF-IDF + LogReg (Subset)"))
    results_llm_subset.append(calculate_metrics(y_test_subset, y_pred_xgb[subset_slice], None, "XGBoost (Subset)"))
    results_llm_subset.append(calculate_metrics(y_test_subset, y_pred_bert[subset_slice], None, "BERT (Subset)"))
    results_llm_subset.append(calculate_metrics(y_test_subset, y_pred_llm, None, f"LLM ({LLM_MODEL})"))

print("Metryki obliczone.")

In [ ]:
df_results_full = pd.DataFrame([{
    "Model": r["model"],
    "Accuracy": f"{r['accuracy']:.4f}",
    "Macro-F1": f"{r['macro_f1']:.4f}",
    "ROC-AUC": f"{r['roc_auc']:.4f}" if r['roc_auc'] else "N/A"
} for r in results_full])

display(df_results_full)

if results_llm_subset:
    df_results_llm = pd.DataFrame([{
        "Model": r["model"],
        "Accuracy": f"{r['accuracy']:.4f}",
        "Macro-F1": f"{r['macro_f1']:.4f}",
        "ROC-AUC": f"{r['roc_auc']:.4f}" if r['roc_auc'] else "N/A"
    } for r in results_llm_subset])
    
    display(df_results_llm)

results_for_plot = results_full + (results_llm_subset if results_llm_subset else [])

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))

plot_data = results_full.copy()
if y_pred_llm is not None:
    llm_res = results_llm_subset[-1]
    plot_data.append(llm_res)

models = [r["model"] for r in plot_data]
f1_scores = [r["macro_f1"] for r in plot_data]
accuracies = [r["accuracy"] for r in plot_data]

x = np.arange(len(models))
width = 0.35

bars1 = ax.bar(x - width/2, f1_scores, width, label='Macro-F1', color='steelblue')
bars2 = ax.bar(x + width/2, accuracies, width, label='Accuracy', color='coral')

ax.set_xlabel('Model')
ax.set_ylabel('Wynik')
ax.set_title('Porównanie Modeli: Detekcja Mowy Nienawiści (Binary)')
ax.set_xticks(x)
ax.set_xticklabels(models, rotation=15, ha='right')
ax.legend()
ax.set_ylim(0, 1)

def add_labels(bars):
    for bar in bars:
        height = bar.get_height()
        ax.annotate(f'{height:.3f}', xy=(bar.get_x() + bar.get_width()/2, height),
                    xytext=(0, 3), textcoords="offset points", ha='center', fontsize=9)

add_labels(bars1)
add_labels(bars2)

plt.tight_layout()
plt.savefig(REPORTS_DIR / "model_comparison.png", dpi=150, bbox_inches="tight")
plt.show()

## 6. Macierze Pomyłek

In [ ]:
n_models = len(results_full)
fig, axes = plt.subplots(1, n_models, figsize=(5*n_models, 5))

if n_models == 1:
    axes = [axes]

for ax, r in zip(axes, results_full):
    cm = confusion_matrix(y_test, r["predictions"], normalize='true')
    disp = ConfusionMatrixDisplay(cm, display_labels=CLASS_NAMES)
    disp.plot(ax=ax, cmap="Blues", values_format=".2f")
    ax.set_title(f"{r['model']}\nMacro-F1: {r['macro_f1']:.4f}")

plt.tight_layout()
plt.savefig(REPORTS_DIR / "confusion_matrices.png", dpi=150, bbox_inches="tight")
plt.show()

## 7. Analiza Błędów (10 przypadków)

In [ ]:
errors = []

for i, (text, true_label) in enumerate(zip(X_test, y_test)):
    preds = {
        "Baseline": int(y_pred_baseline[i]),
        "XGBoost": int(y_pred_xgb[i]),
        "BERT": int(y_pred_bert[i]),
    }
    
    if y_pred_llm is not None and i < len(y_pred_llm):
        preds["LLM"] = int(y_pred_llm[i])
    
    wrong_count = sum(1 for p in preds.values() if p != true_label)
    
    if wrong_count > 0:
        errors.append({
            "index": i,
            "text": text[:300] + "..." if len(text) > 300 else text,
            "true_label": int(true_label),
            "true_class": CLASS_NAMES[true_label],
            "predictions": preds,
            "wrong_count": wrong_count
        })

errors_sorted = sorted(errors, key=lambda x: -x["wrong_count"])

print(f"Razem błędnych klasyfikacji (wśród analizowanych): {len(errors)}")
print(f"\nNajbardziej mylące przypadki:")

In [ ]:
error_analysis = []

for j, err in enumerate(errors_sorted[:10]):
    print(f"\n{'='*70}")
    print(f"Przypadek {j+1}: Prawda = {err['true_class']} ({err['true_label']})")
    print(f"Predykcje: {err['predictions']}")
    print(f"Błędnych modeli: {err['wrong_count']}")
    print(f"Tekst: {err['text']}")
    
    error_analysis.append({
        "case": j+1,
        "true_label": err['true_class'],
        "predictions": str(err['predictions']),
        "text_preview": err['text'][:100] + "...",
        "analysis": ""
    })

df_errors = pd.DataFrame(error_analysis)
df_errors.to_csv(REPORTS_DIR / "error_analysis.csv", index=False)
print(f"\n\nZapisano analizę błędów do {REPORTS_DIR / 'error_analysis.csv'}")

## 8. Zapis Końcowych Wyników

In [ ]:
final_results = {
    "task": "Binary toxicity classification",
    "dataset": "Jigsaw Toxic Comment (Wikipedia)",
    "test_sample_size": len(y_test),
    "seed": SEED,
    "class_names": CLASS_NAMES,
    "models_full_test": [{
        "name": r["model"],
        "accuracy": float(r["accuracy"]),
        "macro_f1": float(r["macro_f1"]),
        "roc_auc": float(r["roc_auc"]) if r["roc_auc"] else None
    } for r in results_full],
    "models_llm_subset": [{
        "name": r["model"],
        "accuracy": float(r["accuracy"]),
        "macro_f1": float(r["macro_f1"]),
        "roc_auc": float(r["roc_auc"]) if r["roc_auc"] else None
    } for r in results_llm_subset]
}

with open(REPORTS_DIR / "final_results.json", "w") as f:
    json.dump(final_results, f, indent=2)

print(f"Zapisano końcowe wyniki do {REPORTS_DIR / 'final_results.json'}")

In [ ]:
print(f"Rozmiar próbki testowej: {len(y_test)}")
print(f"Rozkład klas: 0={sum(y_test==0)}, 1={sum(y_test==1)}, 2={sum(y_test==2)}")
print("\nWyniki (sortowane po Macro-F1):")
for r in sorted(results, key=lambda x: -x['macro_f1']):
    auc_str = f", AUC={r['roc_auc']:.4f}" if r['roc_auc'] else ""
    print(f"  {r['model']:25s} Acc={r['accuracy']:.4f}, F1={r['macro_f1']:.4f}{auc_str}")
print("\nZapisane pliki:")
print(f"  - {REPORTS_DIR / 'model_comparison.png'}")
print(f"  - {REPORTS_DIR / 'confusion_matrices.png'}")
print(f"  - {REPORTS_DIR / 'error_analysis.csv'}")
print(f"  - {REPORTS_DIR / 'final_results.json'}")